# Ejercicio: Extracción de datos de la API de TMDB (The Movie Database)

![imagen](https://www.themoviedb.org/assets/2/v4/logos/v2/blue_square_2-d537fb228cf3ded904ef09b136fe3fec72548ebc1fea3fbbd1ad9e36364db38b.svg)

Trabajaremos con **TMDB**, una de las bases de datos de cine más importantes del mundo. El objetivo es extraer información de películas para realizar un análisis de mercado inicial.

### 1. Registro y Documentación
Tendrás que [registrarte en TMDB](https://www.themoviedb.org/signup) para obtener tu **API Key (v3 auth)** y consultar la [documentación oficial](https://developer.themoviedb.org/reference/intro/getting-started).

### 2. Objetivo del Ejercicio
Queremos que consultes la API para que te devuelva la información de las películas que empiecen por la **inicial de tu nombre** (parámetro `query`). 

Debes guardar la información en un archivo `.csv` con la siguiente estructura de columnas:

| Columna | Descripción |
| :--- | :--- |
| **id** | ID interno de la película en TMDB |
| **title** | Título de la película |
| **release_date** | Fecha de estreno |
| **genres** | Nombres de los géneros (ej: "Acción, Comedia") |
| **vote_average** | Puntuación media de los usuarios |
| **overview** | Sinopsis o resumen de la trama |

### 3. El Reto: Mapeo de Géneros
A diferencia de otras APIs, el endpoint de búsqueda de películas devuelve los géneros como una lista de IDs numéricos (ej: `[28, 12]`). 

**Tu labor es:**
1. Consultar el endpoint de "Genre List" para obtener la relación entre IDs y nombres.
2. Sustituir los IDs en tu DataFrame final por los nombres reales de los géneros (separados por comas).

---

### Código de inicio
Aquí tienes el bloque para empezar a configurar tus llamadas:

```python
import requests
import pandas as pd

# Rellena estas variables
api_key = "TU_API_KEY_AQUÍ"
url_base = "[https://api.themoviedb.org/3](https://api.themoviedb.org/3)"
mi_inicial = "P" # Sustituye por tu inicial

In [2]:
import requests
import pandas as pd

# Configuración
api_key = "930590f441225ca054ca5c0d0a53220e"
url_base = "https://api.themoviedb.org/3"
mi_inicial = "u" # Cambiar por la inicial del alumno
genres = '/genre/movie/list?'
search = '/search/movie?'


In [67]:
# 1. Obtener el diccionario de géneros (ID -> Nombre)
# Tip: Endpoint /genre/movie/list
def get_genres_map(api_key):
    url = url_base + genres + 'api_key=' + api_key
    response = requests.get(url)

    if response.status_code == 200:  # Código 200 indica una respuesta exitosa.
        data = response.json()  # Analizar la respuesta JSON.
    else:
        print("Error en la solicitud: ", response.status_code)
        return
    
    # Guardamos los id de géneros
    generos = {}

    for genre in data['genres']:
        # nombre = genre['name']
        generos[genre['id']] = genre['name']

    return generos


# 2. Buscar películas
# Tip: Endpoint /search/movie
def search_movies(api_key, query, page = 1):
    # Metemos paginado para recuperar las películas de cada página
    url = url_base + search + 'query=' + query + '&page=' + str(page) + '&api_key=' + api_key
    # print(url) # imprimimos para pruebas
    response = requests.get(url)

    if response.status_code == 200:  # Código 200 indica una respuesta exitosa.
        data = response.json()  # Analizar la respuesta JSON.
        return data
    else:
        print("Error en la solicitud: ", response.status_code)
        return


# 3. Procesamiento y Limpieza
def busca_pelis(api_key, query):
    # Búsqueda inicial de películas
    data = search_movies(api_key, query)
    # Recuperamos los géneros de películas
    genres_dict = get_genres_map(api_key)
    
    # Recuperamos total páginas para saber si hay más de una
    pages = data['total_pages']
    lista_pelis = []
    if pages > 1:
        # Bucle de páginas
        for pag in range(pages):
            data = search_movies(api_key, query, page=pag + 1) # range(pages) comienza desde 0 hasta page - 1, sumamos 1
            lista_pelis = guardar_pelis(data, lista_pelis, genres_dict)
    
    else: # Si solo hay una página
        lista_pelis = guardar_pelis(data, lista_pelis)

            
    return lista_pelis


def guardar_pelis(movies, lista_pelis, genres_dict):
    # Bucle por película de la página
    for peli in movies['results']:
        pel = {
            'id' : 0,
            'title' : '',
            'release_date' : '',
            'genres' : [],
            'vote_average' : 0.0,
            'overview' : ''
        }
        pel['id'] = peli['id']
        pel['title'] = peli['title']
        pel['release_date'] = peli['release_date']
        # - Mapear los IDs de géneros a nombres reales.
        for genre in peli['genre_ids']: # Lista géneros
            genero = genres_dict[genre]
            pel['genres'].append(genero)
        pel['vote_average'] = peli['vote_average']
        pel['overview'] = peli['overview']
        lista_pelis.append(pel)
    return lista_pelis


def limpiar_lista_pelis(lista_pelis, mi_inicial):
    lista_limpia = []
    for movie in lista_pelis:
        if movie['title'][0] not in ('U','u'): # title: 'U...'
            continue
        
        lista_limpia.append(movie)
    return lista_limpia


# - Crear el DataFrame.
def lista2df(lista):
    df = pd.DataFrame(lista)
    return df


# - Exportar a CSV.
def export_csv(df, nombre):
    df.to_csv('./data/' + nombre + '.csv', index=False)

In [ ]:
# Pruebo el buscador de géneros
genres_dict = get_genres_map(api_key)
genres_dict

In [ ]:
# Creamos lista vacía y metemos las películas
lista_pelis = []
lista_pelis = busca_pelis(api_key, mi_inicial)
lista_pelis

[{'id': 39405,
  'title': 'U',
  'release_date': '2006-10-11',
  'genres': ['Animation', 'Fantasy'],
  'vote_average': 6.6,
  'overview': "Since her parents passed away, Princess Mona lives by herself in a castle with two hideous, ghastly characters, Goomi and His Lordship. One day her sobs draw the attention of a unicorn, called U, who wants to comfort and protect Mona for as long as she needs. U becomes Mona's companion, her confidant and her inseparable friend. While Mona grows into a beautiful princess, a group of charming, peaceful and fanciful Yeah-Yeah’s settles in the neighboring forest. They have no particular special powers, yet they soon bring about major changes in everyone's lives, especially in Kulka, the dreamy musician."},
 {'id': 1499984,
  'title': 'Gladiator Underground',
  'release_date': '2025-08-22',
  'genres': ['Action'],
  'vote_average': 7.4,
  'overview': "Two brothers compete in the world's deadliest underground fighting tournament, where they must decide wh

In [ ]:
# Limpiamos lista para quedarnos solamente con las que comiencen por 'U'
pelis_limpio = limpiar_lista_pelis(lista_pelis,mi_inicial)


In [5]:
# Comprobamos cuántos registros hay
len(pelis_limpio)

NameError: name 'pelis_limpio' is not defined

In [ ]:
# Visualizamos algunos datos
pelis_limpio

[{'id': 39405,
  'title': 'U',
  'release_date': '2006-10-11',
  'genres': ['Animation', 'Fantasy'],
  'vote_average': 6.6,
  'overview': "Since her parents passed away, Princess Mona lives by herself in a castle with two hideous, ghastly characters, Goomi and His Lordship. One day her sobs draw the attention of a unicorn, called U, who wants to comfort and protect Mona for as long as she needs. U becomes Mona's companion, her confidant and her inseparable friend. While Mona grows into a beautiful princess, a group of charming, peaceful and fanciful Yeah-Yeah’s settles in the neighboring forest. They have no particular special powers, yet they soon bring about major changes in everyone's lives, especially in Kulka, the dreamy musician."},
 {'id': 1303795,
  'title': 'U',
  'release_date': '2024-05-24',
  'genres': ['Science Fiction'],
  'vote_average': 7.0,
  'overview': 'The storms we go through help us deconstruct to rebuild.'},
 {'id': 1655272,
  'title': 'U',
  'release_date': '202

In [64]:
# Convertimos a df
df_pelis = lista2df(pelis_limpio)
df_pelis

,id,title,release_date,genres,vote_average,overview
0,39405,U,2006-10-11,"[Animation, Fantasy]",6.6,"Since her parents passed away, Princess Mona l..."
1,1303795,U,2024-05-24,[Science Fiction],7.0,The storms we go through help us deconstruct t...
2,1655272,U,2026-03-20,[Music],0.0,U is a album video that accompanies the Americ...
3,1510235,Udaipur Files: Kanhaiya Lal Tailor Murder,2025-08-08,"[Crime, Drama]",7.0,"In June 2022, Kanhaiya Lal Sahu, a tailor in U..."
4,1132364,U,2002-12-30,[],8.4,A short movie by yuri A containing multiple po...
...,...,...,...,...,...,...
3454,1418551,Unbearably Loud,2024-04-17,[Documentary],0.0,This film tells the stories of five Russian fa...
3455,1297660,Us and Our Elders,1952-01-01,[Documentary],0.0,Instructive pictures that show how to try to s...
3456,349343,Ulysses,1999-01-09,[],0.0,
3457,991932,Uh-puh!,2022-07-08,[],0.0,"Danbi, who was notified of her breakup by her ..."


In [ ]:
# Exportamos el df a un .csv con el nombre 'peliculas_u.csv'
export_csv(df_pelis,'peliculas_u')

In [ ]:
# Leemos el .csv para comprobar que el archivo está bien creado
pd.read_csv("data/peliculas_u.csv")

,id,title,release_date,genres,vote_average,overview
0,39405,U,2006-10-11,"['Animation', 'Fantasy']",6.6,"Since her parents passed away, Princess Mona l..."
1,1303795,U,2024-05-24,['Science Fiction'],7.0,The storms we go through help us deconstruct t...
2,1655272,U,2026-03-20,['Music'],0.0,U is a album video that accompanies the Americ...
3,1510235,Udaipur Files: Kanhaiya Lal Tailor Murder,2025-08-08,"['Crime', 'Drama']",7.0,"In June 2022, Kanhaiya Lal Sahu, a tailor in U..."
4,1132364,U,2002-12-30,[],8.4,A short movie by yuri A containing multiple po...
...,...,...,...,...,...,...
3454,1418551,Unbearably Loud,2024-04-17,['Documentary'],0.0,This film tells the stories of five Russian fa...
3455,1297660,Us and Our Elders,1952-01-01,['Documentary'],0.0,Instructive pictures that show how to try to s...
3456,349343,Ulysses,1999-01-09,[],0.0,NaN
3457,991932,Uh-puh!,2022-07-08,[],0.0,"Danbi, who was notified of her breakup by her ..."
